# Neural Sequence Model: Char-CNN BiLSTM-CRF

In this notebook, I implement a more sophisticated architecture for the NER task, specifically following the Ma and Hovy (2016) approach. I decided to move away from a basic BiLSTM + Softmax setup because NER requires a better way to handle rare words and label consistency.

- Character embeddings for each word
- CNN over characters to create a character-level word representation
- Pretrained GloVe word embeddings
- Hybrid Embeddings concatenating those CNN-generated character representations with pretrained GloVe word embeddings.
- BiLSTM to read left and right context
- CRF output layer to decode the best label sequence

This architecture is useful for NER because named entities often contain rare or unseen words(OOV words). The character CNN handles the variety, while the CRF ensures the final output is a sequence that actually makes sense.

### **1. Setup**

In [2]:
import sys
import os
from collections import Counter
import random
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

sys.path.append(os.path.abspath(".."))

### **2. Load Data**

In [3]:
from src.load_dataset import load_dataset

train_sents, train_tags = load_dataset("../data/raw/train.txt")
valid_sents, valid_tags = load_dataset("../data/raw/valid.txt")
test_sents, test_tags = load_dataset("../data/raw/test.txt")

print(f"Training sentences: {len(train_sents)}")
print(f"Valid sentences: {len(valid_sents)}")
print(f"Testing sentences: {len(test_sents)}")

Training sentences: 14041
Valid sentences: 3250
Testing sentences: 3453


In [4]:
print(train_sents[:5])
print(valid_tags[:5])
print(test_sents[:5])

[['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], ['Peter', 'Blackburn'], ['BRUSSELS', '1996-08-22'], ['The', 'European', 'Commission', 'said', 'on', 'Thursday', 'it', 'disagreed', 'with', 'German', 'advice', 'to', 'consumers', 'to', 'shun', 'British', 'lamb', 'until', 'scientists', 'determine', 'whether', 'mad', 'cow', 'disease', 'can', 'be', 'transmitted', 'to', 'sheep', '.'], ['Germany', "'s", 'representative', 'to', 'the', 'European', 'Union', "'s", 'veterinary', 'committee', 'Werner', 'Zwingmann', 'said', 'on', 'Wednesday', 'consumers', 'should', 'buy', 'sheepmeat', 'from', 'countries', 'other', 'than', 'Britain', 'until', 'the', 'scientific', 'advice', 'was', 'clearer', '.']]
[['O', 'O', 'B-ORG', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], ['B-LOC', 'O'], ['B-MISC', 'I-MISC', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORG', 'O', 'B-ORG', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],

### **3. Build Word, Character, and Label Vocabularies**

The model needs three vocabularies:

- a word vocabulary for word embeddings
- a character vocabulary for the character CNN
- a tag vocabulary for the NER labels

I lowercase words for the word vocabulary to improve coverage with pretrained embeddings. However, I still preserve the original token form when building character inputs. This is important because the character CNN is meant to capture subword and morphological signals such as capitalisation, suffixes, punctuation, and word shape.


In [5]:
# To count each word appears in the training set
word_counter = Counter(word.lower() for sent in train_sents for word in sent)

# To count each character appears in the training set
char_counter = Counter(ch for sent in train_sents for word in sent for ch in word)

# To extract sorted tag list from the training set
tag_list = sorted(set(tag for sent_tags in train_tags for tag in sent_tags))

I sorted the tags to keep the ID mapping consistent. Since sets are unordered, sorting ensures the same tag gets the same index every time I run the code.

In [6]:
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'
PAD_CHAR = '<PAD>'
UNK_CHAR = '<UNK>'


# Initialize dictionaries with PAD_TOKEN and UNK_TOKEN
word_to_id = {PAD_TOKEN: 0, UNK_TOKEN: 1}
char_to_id = {PAD_CHAR: 0, UNK_CHAR: 1}

# Append each word to word_to_id and give ID
for word, count in word_counter.items():
    word_to_id[word] = len(word_to_id)

# Append each word to char_to_id and give ID
for ch, count in char_counter.items():
    char_to_id[ch] = len(char_to_id)

# Assign an ID for each tag
tag_to_id = {tag: idx for idx, tag in enumerate(tag_list)}


In [7]:
print(f'Word vocab size: {len(word_to_id):}')
print(f'Char vocab size: {len(char_to_id):}')
print(f'Number of tags: {len(tag_to_id)}')
print(tag_to_id)

Word vocab size: 21011
Char vocab size: 86
Number of tags: 9
{'B-LOC': 0, 'B-MISC': 1, 'B-ORG': 2, 'B-PER': 3, 'I-LOC': 4, 'I-MISC': 5, 'I-ORG': 6, 'I-PER': 7, 'O': 8}


I convert words,characters and tags into IDs because the model cannot work with text, only numbers. 

I set "0" **<PAD>** to handle padding when sentences have different lengths, and set it to 0 so it can be ignored during training. 

Secondly, add **<UNK>** to handle unseen words or characters, and set it to 1 , so the model can still learn a representation for unknown inputs. 

Lastly, I assign a unique ID to each word, character and tags from the training data so everything converted into numerical form for the model.

### **4. Dataset and Padding**

This step is all about organizing our raw lists into structured **batches**.

In [ ]:
# Converts the word into the index
def word_to_index(word):
    # Assigned default value to 1, 
    # If word not in dictionary treat it UNKNOWN WORD
    return word_to_id.get(word.lower(), word_to_id[UNK_TOKEN])

# Converts the character into the index 
# and create the sentence with char ID's
def chars_to_indices(word):
    return [char_to_id.get(ch, char_to_id[UNK_CHAR]) for ch in word]

# Pytorch wrapper
class NERDataset(Dataset):
    def __init__(self, sentences, tags):
        self.sentences = sentences
        self.tags = tags

    # Define to access length of sentences
    def __len__(self):
        return len(self.sentences)

    # Define to access tags and sentences using indexing
    def __getitem__(self, idx):
        words = self.sentences[idx]
        tags = self.tags[idx]
        
        word_ids = [word_to_index(word) for word in words] # Extract word index
        char_ids = [chars_to_indices(word) for word in words] # Extract char index
        tag_ids = [tag_to_id[tag] for tag in tags] # # Extract tag index

        return {'words': words, # word
                'word_ids': word_ids, # word id, input for word embedding
                'char_ids': char_ids, # char id, input for char embedding
                'tag_ids': tag_ids # tag id
                }

In [ ]:
from src.collate_batches import collate_batches
from functools import partial

# Creates collate_fn function assigned tag_to_id parameter
collate_fn = partial(collate_batches, tag_to_id=tag_to_id)

batch_size = 32 # Initialize batch size

# Create train loader, validation loader and test loader
train_loader = DataLoader(NERDataset(train_sents, train_tags), batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(NERDataset(valid_sents, valid_tags), batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(NERDataset(test_sents, test_tags), batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

I create the NERDataset class as a PyTorch wrapper. Inside `__getitem__`, I take one sentence and convert its words, characters, and tags into their corresponding IDs and it return everything in a dictionary.

After that, implement a custom collate_batches function to handle batching and padding. I use batch size 32, practical choice that works well in most NLP tasks and it is a good balance between speed and memory. I also use dynamic padding so that each batch is padded only to its maximum length, which makes training more efficient and avoids unnecessary padding. 

The mask ensures that padded positions do not affect the model. This is important because the BiLSTM and CRF layers should only learn from actual tokens, not padded values.

Lastly, I shuffle the training data to help the model generalise better, but I do not shuffle validation and test data so the evaluation stays consistent.

In [73]:
example_batch = next(iter(train_loader))
print(f"word_ids: {example_batch['word_ids'].shape}")
print(f"char_ids: {example_batch['char_ids'].shape}")
print(f"tag_ids: {example_batch['tag_ids'].shape}")
print(f"mask: {example_batch['mask'].shape}")

word_ids: torch.Size([32, 43])
char_ids: torch.Size([32, 43, 14])
tag_ids: torch.Size([32, 43])
mask: torch.Size([32, 43])


In [38]:
len(example_batch['words'][0])

2

In [75]:
for i in range(3):
      
    print(f"sentence: {example_batch['words'][i]}")
    print(example_batch['mask'][i])
    print(example_batch['word_ids'][i])
    print(example_batch['char_ids'][i])

sentence: ['-', 'IP', '/', '96', '/', '805', ':', 'Commission', 'finds', 'acquisition', 'of', 'CAMAT', 'by', 'AGF-IART', 'does', 'not', 'fall', 'under', 'the', 'merger', 'regulation', '.']
tensor([ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False])
tensor([  637, 18773,   441,  1541,   441, 18777,   377,    17,  3662,  9084,
          160, 18778,    92, 18779,  1205,   787,  2122,   121,    15, 12405,
         3427,    10,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0])
tensor([[31,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [61, 22,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, 

In [76]:
example_batch['tag_ids'][:5]

tensor([[8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 2, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8,
         8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8],
        [3, 7, 8, 0, 8, 8, 8, 8, 3, 7, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8,
         8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8],
        [8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 7, 8, 8, 0, 8, 8, 3, 8,
         8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8],
        [8, 8, 8, 2, 6, 6, 8, 8, 8, 8, 8, 8, 3, 7, 8, 8, 8, 0, 8, 8, 8, 8, 8, 8,
         8, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8],
        [0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8,
         8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]])

### **5. Pretrained GloVe word embeddings**

I use pretrained GloVe embeddings to provide the model with a solid foundation of word meanings learned from a massive external corpus. This is a crucial step for my project because the initial OOV analysis revealed a significant gap. About 12.18% of words in the test set and 8.36% in the validation set were never seen during training. 

When I look at entity types, 

-   the PER category has a very high OOV rate, reaching 54.02% in the test set. This means many person names are not seen during training. 
-   ORG entities also have relatively high OOV rates 23.32%, showing that some categories are more sparse than others.

Because of this, if the model only relies on training data, it may struggle with unseen words. Pretrained embeddings help solve this by placing similar words closer in the embedding space, so the model can generalise better.

When it comes to the "why I choose GloVe instead of Word2Vec",  because it uses global word co-occurrence information, which gives more stable semantic representations. Also, GloVe embeddings are widely available in standard formats and are easy to use.

Furthermore, based on the dataset statistics, 

- The training set contains around 203k tokens across approximately 14k sentences, indicating a small to moderate size dataset. 
- Sentence lengths are relatively short, with a median of around 9–11 tokens and 95% of sentences below 40 tokens.

This analysis guides my choice of embedding dimension.

Using very high-dimensional embeddings, softened the model with unnecessary extra parameters. Since the dataset is not very large, I use 100d embedding as they provide enough capacity to represent word meaning while keeping the model efficient and reducing the risk of overfitting.


In [47]:
with open("../data/glove/wiki_giga_2024_100.txt", encoding="utf-8") as f:
    for i in range(5):
        print(f.readline())

the 0.306717 -0.32053 -0.39364699999999997 0.08282600000000001 0.073522 -0.409154 -0.265564 -0.23693999999999998 -0.305832 0.74529 0.214341 0.27678099999999994 -0.152797 -0.127524 0.11952500000000002 0.640965 -0.175869 0.160711 0.47797799999999996 -0.160939 -0.150093 0.674601 -0.099565 0.021881999999999985 -0.032770999999999995 0.368641 -0.08701900000000007 -0.13332599999999997 0.170143 0.15693399999999996 0.6775059999999999 -0.099686 0.392113 0.37343400000000004 -5.736062 0.413845 0.477368 -0.04169700000000001 0.38310900000000003 0.12015199999999998 -0.20947 0.605104 0.23635299999999998 0.15113100000000002 -0.508865 0.671239 -0.300263 -0.267927 2.549487 0.06717699999999999 0.217224 -0.031316 0.05231 0.119321 -0.332154 -0.8079040000000001 -0.546453 -0.04439199999999999 -0.281657 0.286647 0.32577500000000004 -0.021960000000000007 -0.636903 -0.268063 0.247956 -0.402493 0.276707 -0.275139 0.20115899999999998 0.08284399999999997 0.591695 -0.017126999999999948 -0.09226899999999999 0.3920079

I initialise embeddings for unseen words using a normal distribution with a small standard deviation 0.5, so that they are similar in scale to pretrained GloVe vectors, which are typically in the range of approximately [-1, 1] with a standard deviation around 0.5–0.6. Instead of assigning identical zero vectors to all unknown words, this helps the model learn meaningful representations. 

`float32`-4 bytes per value- is used instead of float64 -8 bytes per value- because it is more efficient in terms of memory and computation. Float64 doubles the memory usage and slows down training. Also, GPUs and Pytorch are optimized for float32.

In [68]:
from src.load_dataset import load_glove

word_embeddings, glove_found = load_glove("../data/glove/wiki_giga_2024_100.txt", word_to_id)

During implementation, I encountered a case where a GloVe vectors (102d) did not match the expected embedding dimension. ValueError occured when constructing the embedding matrix. To handle this, I added a dimension check to ensure only valid vectors are used.

In [69]:
vocab = len(word_to_id)
print(f"Matched GloVe vectors: {glove_found}")
print(f"Coverage: {(glove_found / vocab):.2%}")

Matched GloVe vectors: 18327
Coverage: 87.23%


The pretrained GloVe embeddings cover a substantial portion of the vocabulary, with 18,327 words successfully matched 87.32%. This suggests that most frequent words benefit from pretrained semantic representations.

However, some words remain unmatched and are initialised randomly, which may affect performance, especially for rare entities such as person names.